In [ ]:
# import the required packages


In [ ]:
# Load the SQL toolset extension
%load_ext sql

In [ ]:
#2. Connect to (and create) the database file
%sql sqlite:///bike_orders_database.sqlite

In [ ]:
def collect_data():
    """
    Collects and combines the bike orders data using the jupyter-sql magic extension.
    """
    
    # 1.0 Connect to database and load tables via SQL magic
    # Using the %sql magic command to fetch data directly into DataFrames
    bikes     = %sql SELECT * FROM bikes
    bikeshops = %sql SELECT * FROM bikeshops
    orderlines = %sql SELECT * FROM orderlines
    
    # Convert to DataFrames and drop 'index' column
    data_dict = {
        'bikes': bikes.DataFrame().drop("index", axis=1),
        'bikeshops': bikeshops.DataFrame().drop("index", axis=1),
        'orderlines': orderlines.DataFrame().drop("index", axis=1)
    }

    # 2.0 Combining Data
    joined_df = data_dict['orderlines'] \
        .merge(
            right    = data_dict['bikes'],
            how      = 'left',
            left_on  = 'product.id',
            right_on = 'bike.id'
        ) \
        .merge(
            right    = data_dict['bikeshops'],
            how      = "left",
            left_on  = "customer.id",
            right_on = 'bikeshop.id'
        )

    # 3.0 Cleaning Data 
    df = joined_df
    df['order.date'] = pd.to_datetime(df['order.date'])

    temp_df = df['description'].str.split(" - ", expand = True)
    df['category.1'] = temp_df[0]
    df['category.2'] = temp_df[1]
    df['frame.material'] = temp_df[2]

    temp_df = df['location'].str.split(", ", expand = True)
    df['city'] = temp_df[0]
    df['state'] = temp_df[1]

    df['total.price'] = df['quantity'] * df['price']

    cols_to_keep_list = [
        'order.id', 'order.line', 'order.date',    
        'quantity', 'price', 'total.price', 
        'model', 'category.1', 'category.2', 'frame.material', 
        'bikeshop.name', 'city', 'state'
    ]

    df = df[cols_to_keep_list]
    df.columns = df.columns.str.replace(".", "_")

    return df

In [ ]:
df = collect_data()

# 1. Selecting Columns

### Select by name
![Select by name](subset_cols.png)

In [ ]:
# Select by name
# Subsetting data frame columns with a list of names


### Select by position
In pandas, *loc* and *iloc* are the primary tools for selecting data from a DataFrame. The fundamental difference lies in how they identify the data: *loc* is label-based, while *iloc* is integer-position-based.
```python
df.loc[row_label, column_label]
df.iloc[row_position, column_position]
```
![Select by position](subset_iloc.png)

In [ ]:
df

In [ ]:
# Select by position
# iloc


# loc
# Select all rows where price is greater than 12000


In [ ]:
# Select by text matching
# Use df.filter().  Filter is based on column or row index using text patterns
# The pandas filter() function is a specialized tool used to subset a DataFrame or Series based on labels 
# rather than values. It is highly useful when you need to select columns or index rows that match 
# specific naming patterns, such as prefixes or suffixes.
# If they try to use it to filter rows based on the content of a column (like filter(quantity > 5)), 
# it will throw an error. For content-based filtering, they should use boolean indexing instead 
# (e.g., df[df['quantity'] > 5]).
# Most of the time I use:
#  "term" = Contains "term"
#  "^term" = Starts with "term"
#  "term$" = Ends with "term"
#  (term1)|(term2) = Matches "term1" or "term2"


In [ ]:
# Rearranging a single column
# Move 'model' to the front


In [ ]:
# Rearranging multiple columns
# Move 'model', 'category_1', and 'category_2' to the front


In [ ]:
# Rearranging multiple columns (list comprehension)



# Step by step list comprehension


In [ ]:
# Select by data types
# df.select_dtypes(): Select columns and subset by dtype (data type)




# Concatenate data frames
# pd.concat(): Combines multiple data frames contained in a list


# Drop columns
# df.drop(): Exclude columns by name


# move ['model', 'category_1', 'category_2'] to the front


# 2. Arranging Rows

In [ ]:
# Sorting
# df.sort_values(): Great for sorting by a column. Can sort by multiple columns to get group-wise sorts.


# Sorting a Series


# 3. Rowwise Filtering (Slicing)

In [ ]:
# Simple filters with booleans
# The key concept is that we need to create a "Boolean Series", a pandas Series that consists only of
# True/False values. We then use this to subset our data frame rowwise



# Use string accessors
# s.str.startswith(): We can use the string accessor of pandas series containing string data.
# The string accessor has startswith(), contains(), and endswith() methods.


In [ ]:
# Query filters
# df.query(): A string-based rowwise filtering method. It allows Boolean expressions for filtering.
# Works well in method chaining.
#  df.query('Length > 7')
# df.query('length > 7 and Width < 8')
# df.query('Name.str.startswith("abc")', engine="python")




In [ ]:
# Filtering items in a list. Filtering with isin() and ~
# series.isin(): Detect whether values are in a list of values



# An alternative to using series.isin() is to use regex pattern with:
# series.str.contains(regex="(Trail) | (Cyclocross)")

# Negates using tilde (~).  It flips the boolean series to form the opposite True/False.



In [ ]:
# Slicing


# Index slicing with df.iloc[]


In [ ]:
# Unique / Distinct Values
# Drop Duplicates
# df.drop_duplicates(): remove duplicate rows returning the unique combinations


In [ ]:
# Top / Bottom
# nlargest(): Finds the N-largest rows using a numeric column


In [ ]:
# Sampling Rows
# df.sample(): Select random rows. Good for train/test splitting.  Use random_state to make reproducible.



# 4. Adding Calculated Columns (Mutating)

In [ ]:
# Close the connection
%sql --close sqlite:///bike_orders_database.sqlite